# A Constraint-Based CKY Parser Implementation for English

## Prepared by
- Mehmet Ali Özdemir
- Tarık Emre Tıraş

## Grammar Class

In [ ]:
import collections

In [ ]:
class Grammar:
    def __init__(self):
        # Stores rules indexed by RHS for CKY.
        # Structure: { ("NP", "VP"): [ {'lhs': 'S', 'validator': <func>, 'features': {}}, ... ] }
        self.rules_by_rhs = collections.defaultdict(list)

        # Stores rules indexed by LHS (useful for debugging or CNF conversion)
        self.rules_by_lhs = collections.defaultdict(list)

        self.terminals = set()
        self.non_terminals = set()

    def fill_rules(self, rules):
        """
        Parses the new SYNTACTIC_RULES format: (LHS, RHS, Validator)
        """
        for rule_tuple in rules:
            # Unpack based on length (handled flexibly)
            if len(rule_tuple) == 3:
                lhs, rhs, validator = rule_tuple
                self.add_rule(lhs, rhs, validator=validator)
            else:
                lhs, rhs = rule_tuple
                self.add_rule(lhs, rhs)

    def add_rule(self, lhs, rhs, validator=None, features=None):
        """
        lhs: String (e.g., "S", "VBD")
        rhs: Tuple of strings (e.g., ("NP", "VP")) or ("word",)
        validator: Function (for Syntactic Rules) -> checks agreement
        features: Dict (for Lexical Rules) -> static features like {'num': 'sg'}
        """
        if not isinstance(rhs, tuple):
            rhs = tuple(rhs)

        # 1. Register Terminals/Non-Terminals
        self.non_terminals.add(lhs)
        if lhs in self.terminals:
            self.terminals.remove(lhs)

        for symbol in rhs:
            # Heuristic: If we haven't seen it as a LHS yet, assume terminal for now
            if symbol not in self.non_terminals:
                self.terminals.add(symbol)

        # 2. Construct the Rule Object
        # We use a dictionary so we can easily expand it later if needed
        rule_obj = {
            'lhs': lhs,
            'rhs': rhs,
            'validator': validator,  # The function to run (e.g., agree_subj_verb)
            'features': features     # Static features (e.g., for lexical rules)
        }

        # 3. Store in Index
        # Check for duplicates to avoid adding the exact same rule twice
        current_rules = self.rules_by_rhs[rhs]
        if not any(r['lhs'] == lhs and r['validator'] == validator for r in current_rules):
            self.rules_by_rhs[rhs].append(rule_obj)

        # Store in LHS index (simplified storage for display/debugging)
        self.rules_by_lhs[lhs].append(rule_obj)

    def get_rules_for_rhs(self, rhs):
        """
        Primary lookup for CKY.
        Returns a list of rule dictionaries: [{'lhs': 'S', 'validator': ...}, ...]
        """
        return self.rules_by_rhs.get(rhs, [])

    def get_rhs(self, lhs):
        """
        Returns list of full rule objects for a given LHS.
        """
        return self.rules_by_lhs.get(lhs, [])

    def is_terminal(self, sym):
        """
        Check if a symbol is a terminal (never appears as LHS).
        """
        return sym not in self.non_terminals

    def print_grammar(self):
        print("--- Grammar Rules ---")
        for lhs, rules in self.rules_by_lhs.items():
            for r in rules:
                rhs_str = " ".join(r['rhs'])
                extra = ""
                if r['validator']:
                    extra += f" [Check: {r['validator'].__name__}]"
                if r['features']:
                    extra += f" [Feats: {r['features']}]"
                print(f"{lhs} -> {rhs_str}{extra}")

## Morphology

In [ ]:
class MorphAnalyzer:
    def __init__(self):
        self.lexicon = {}
        self._build_lexicon()

    def add_word(self, word, pos, features=None):
        """Helper to register words with features."""
        if features is None:
            features = {}
        if word not in self.lexicon:
            self.lexicon[word] = []
        # Store as tuple: (POS_TAG, FEATURE_DICT)
        self.lexicon[word].append((pos, features))

    def _build_lexicon(self):
        # --- NOUNS ---
        # Singular
        singular_nouns = [
            "friend", "mother", "dinner", "culture", "history", "watermelon",
            "fruit", "summer", "meeting", "moonlight", "tree", "school",
            "village", "music", "book", "gift", "present", "father"
        ]
        for w in singular_nouns:
            self.add_word(w, "NN", {"num": "sg", "per": 3})

        # Time Nouns (Semantic feature for "every night")
        self.add_word("night", "NN", {"num": "sg", "per": 3, "sem": "time"})

        # Plural
        plural_nouns = ["friends", "novels", "epics"]
        for w in plural_nouns:
            self.add_word(w, "NNS", {"num": "pl", "per": 3})

        # --- PRONOUNS ---
        for w in ["he", "she", "it"]:
            self.add_word(w, "PRP", {"num": "sg", "per": 3, "case": "nom"})

        self.add_word("i",    "PRP", {"num": "sg", "per": 1, "case": "nom"})
        self.add_word("we",   "PRP", {"num": "pl", "per": 1, "case": "nom"})
        self.add_word("they", "PRP", {"num": "pl", "per": 3, "case": "nom"})
        self.add_word("you",  "PRP", {"per": 2, "case": "nom"})

        for w in ["my", "our", "your"]:
            self.add_word(w, "PRP$", {})

        # --- VERBS ---
        # 1. Past (VBD)
        # Note: "read" is also VBD (past)
        for w in ["bought", "helped", "watched", "booked", "read"]:
            self.add_word(w, "VBD", {"tense": "past", "subcat": "tr"})

        self.add_word("went", "VBD", {"tense": "past", "subcat": "intr"})
        self.add_word("was", "VBD", {"tense": "past", "num": "sg", "subcat": "cop"})
        self.add_word("did", "VBD", {"tense": "past", "subcat": "aux"})

        # 2. Present 3rd Sg (VBZ)
        for w in ["buys", "reads", "helps"]:
            self.add_word(w, "VBZ", {"tense": "pres", "num": "sg", "per": 3, "subcat": "tr"})
        self.add_word("goes", "VBZ", {"tense": "pres", "num": "sg", "per": 3, "subcat": "intr"})
        self.add_word("is", "VBZ", {"tense": "pres", "num": "sg", "per": 3, "subcat": "cop"})

        # 3. Present Plural/Base (VBP)
        # Includes "tell", "read"
        for w in ["enjoy", "buy", "do", "tell", "read"]:
             self.add_word(w, "VBP", {"tense": "pres", "num": "pl", "subcat": "tr"})
        self.add_word("go", "VBP", {"tense": "pres", "num": "pl", "subcat": "intr"})

        self.add_word("tell", "VBP", {"tense": "pres", "num": "pl", "subcat": "intr"})
        self.add_word("go", "VBP", {"tense": "pres", "num": "pl", "subcat": "intr"})

        # 4. Base Form (VB)
        for w in ["attend", "tell", "help", "read", "do"]:
             self.add_word(w, "VB", {"tense": "base", "subcat": "tr"})
        for w in ["come", "go", "listen"]:
             self.add_word(w, "VB", {"tense": "base", "subcat": "intr"})

        # --- DETERMINERS (FIX: Add Number) ---
        for w in ["a", "an", "this"]:
            self.add_word(w, "DT", {"num": "sg"})
        for w in ["these", "those"]:
            self.add_word(w, "DT", {"num": "pl"})
        for w in ["the", "every"]: # 'every' is technically sg, but 'the' is neutral
            self.add_word(w, "DT", {})

        # --- OTHERS ---
        for w in ["not", "quite", "far", "here", "lastly", "yesterday", "tonight"]: self.add_word(w, "RB")
        self.add_word("when", "WRB")
        self.add_word("most", "RBS") # Superlative
        self.add_word("will", "MD")
        self.add_word("can", "MD")
        for w in ["historical", "national", "beautiful", "loud"]: self.add_word(w, "JJ")
        self.add_word("and", "CC")

        # Prepositions
        self.add_word("to", "TO", {"type": "dir"})
        self.add_word("to", "IN", {"type": "dir"})
        self.add_word("from", "IN", {"type": "dir"})
        for w in ["in", "under", "at"]: self.add_word(w, "IN", {"type": "loc"})
        for w in ["for", "with", "about", "of"]: self.add_word(w, "IN", {"type": "gen"})

    def get_tags(self, word):
        """
        Returns a list of (POS, FeatureDict) tuples.
        Example: "bought" -> [('VBD', {'tense': 'past', 'subcat': 'tr'})]
        """
        return self.lexicon.get(word.lower(), [])

    def feed_to_grammar(self, grammar):
        """
        Registers words in the grammar.
        FIX: Explicitly pass 'features' as a keyword argument.
        """
        for word, entries in self.lexicon.items():
            for pos, features in entries:
                # Pass features explicitly to the correct parameter
                grammar.add_rule(pos, (word,), features=features)

## Syntax Constraints

In [ ]:
# --- CONSTRAINT & PROPAGATION HELPERS ---

def agree_subj_verb(rhs_nodes):
    """ Rule: S -> NP VP """
    np, vp = rhs_nodes[0], rhs_nodes[1]

    np_num = np['feats'].get('num')
    vp_num = vp['feats'].get('num')
    np_per = np['feats'].get('per')
    vp_per = vp['feats'].get('per')

    # 1. Number Agreement
    if np_num is not None and vp_num is not None:
        if np_per == 1 and np_num == 'sg' and vp_num == 'pl':
            pass # Exception for "I"
        elif np_num != vp_num:
            return None

    # 2. Person Agreement
    if np_per is not None and vp_per is not None:
        if np_per != vp_per:
            return None

    # CRITICAL FIX: Propagate Tense!
    # This allows 'S -> MD S_SUBJ_VP' to check if the inner VP is Base form.
    return {'type': 'decl', 'tense': vp['feats'].get('tense')}

def require_transitive(rhs_nodes):
    """ VP -> VBD NP """
    verb = rhs_nodes[0]
    if verb['feats'].get('subcat') != 'tr':
        return None
    # FIX: Propagate 'per' so we can reject "I buys" (buys is 3rd per)
    return {'tense': verb['feats'].get('tense'),
            'num': verb['feats'].get('num'),
            'per': verb['feats'].get('per')}

def require_intransitive_pp(rhs_nodes):
    verb, pp = rhs_nodes[0], rhs_nodes[1]
    if verb['feats'].get('subcat') != 'intr': return None
    if pp['feats'].get('type') != 'dir': return None
    return {'tense': verb['feats'].get('tense'),
            'num': verb['feats'].get('num'),
            'per': verb['feats'].get('per')}

def require_intransitive_bare(rhs_nodes):
    verb = rhs_nodes[0]
    if verb['feats'].get('subcat') != 'intr': return None
    return {'tense': verb['feats'].get('tense'),
            'num': verb['feats'].get('num'),
            'per': verb['feats'].get('per')}

def require_copula(rhs_nodes):
    verb = rhs_nodes[0]
    if verb['feats'].get('subcat') != 'cop': return None
    return {'tense': verb['feats'].get('tense'),
            'num': verb['feats'].get('num'),
            'per': verb['feats'].get('per')}

def agree_det_noun(rhs_nodes):
    """ NP -> DT NN """
    det, noun = rhs_nodes[0], rhs_nodes[1]
    det_num = det['feats'].get('num')
    noun_num = noun['feats'].get('num')

    # Fix: Reject "a novels" (sg vs pl)
    if det_num is not None and noun_num is not None:
        if det_num != noun_num:
            return None

    # Inherit semantic features (e.g. time)
    return {'num': noun_num, 'per': 3, 'sem': noun['feats'].get('sem')}

def pass_head_features(rhs_nodes):
    return rhs_nodes[0]['feats'].copy()

def check_time_adjunct(rhs_nodes):
    """ VP -> VP NP (Only if NP is time, e.g. 'every night') """
    np = rhs_nodes[1]
    if np['feats'].get('sem') == 'time':
        return rhs_nodes[0]['feats'].copy()
    return None

def pass_tail_features(rhs_nodes):
    """
    For rules like 'JJ NN', the head (Noun) is the LAST child.
    We must inherit features from the Noun, not the Adjective.
    """
    return rhs_nodes[-1]['feats'].copy()

## Syntax

In [ ]:
SYNTACTIC_RULES = [
    # --- 1. SENTENCE LEVEL ---
    ("S", ("NP", "VP"), agree_subj_verb),
    ("S", ("VP",), lambda rhs: {'type': 'imp'} if rhs[0]['feats'].get('tense') == 'base' else None),

    ("VP_DO_NOT", ("VB", "RB"), lambda rhs: {'type': 'neg_aux'} if rhs[0]['word'] == 'do' and rhs[1]['word'] == 'not' else None),
    ("S", ("VP_DO_NOT", "VP"), lambda rhs: {'type': 'imp_neg'} if rhs[1]['feats'].get('tense') == 'base' else None),

    # Questions
    # Update: S_SUBJ_VP uses normal agreement, but now returns 'tense'
    ("S_SUBJ_VP", ("NP", "VP"), agree_subj_verb),

    # Update: Check that the body of the question is Base Tense (e.g., "Can he go", not "Can he went")
    ("S", ("MD", "S_SUBJ_VP"),
     lambda rhs: {'type': 'q_yn'} if rhs[1].get('feats', {}).get('tense') == 'base' else None),

    ("S_Q_HEAD", ("WRB", "VBD"), lambda rhs: {'type': 'wh_aux'} if rhs[1]['feats'].get('subcat') == 'aux' else None),
    ("S", ("S_Q_HEAD", "S_SUBJ_VP"), lambda rhs: {'type': 'q_wh'} if rhs[0]['feats'].get('type') == 'wh_aux' else None),


    # --- 2. NOUN PHRASES ---
    ("NP", ("PRP",), pass_head_features),
    ("NP", ("NN",), pass_head_features),
    ("NP", ("NNS",), pass_head_features),
    ("NP", ("DT", "NN"), agree_det_noun),
    ("NP", ("DT", "NNS"), agree_det_noun),

    # CRITICAL FIX: Use pass_tail_features for Adjective + Noun
    ("NP_ADJ_N", ("JJ", "NN"), pass_tail_features),
    ("NP_ADJ_N", ("JJ", "NNS"), pass_tail_features),

    ("ADJP", ("RBS", "JJ"), lambda rhs: {'type': 'superlative'}),
    # "most beautiful fruit" -> head is 'fruit' (NN), which is at index 1 (tail)
    ("NP_ADJ_N", ("ADJP", "NN"), pass_tail_features),

    ("NP", ("DT", "NP_ADJ_N"), agree_det_noun),
    ("NP", ("NP_ADJ_N",), pass_head_features),

    ("NP", ("NP", "PP"), pass_head_features),
    ("NP", ("PRP$", "NN"), pass_head_features),
    ("NP", ("PRP$", "NNS"), pass_head_features),


    # --- 3. VERB PHRASES ---
    ("VP", ("VBD", "NP"), require_transitive),
    ("VP", ("VBZ", "NP"), require_transitive),
    ("VP", ("VBP", "NP"), require_transitive),
    ("VP", ("VB", "NP"),  require_transitive),

    ("VP", ("VBD", "PP"), require_intransitive_pp),
    ("VP", ("VB", "PP"),  require_intransitive_pp),
    ("VP", ("VB",), require_intransitive_bare),
    ("VP", ("VBP",), require_intransitive_bare),

    ("VP", ("VBD", "JJ"), require_copula),
    ("VP", ("VBD", "RB"), require_copula),
    ("AP", ("RB", "JJ"), lambda rhs: {'type': 'adj_phrase'}),
    ("VP", ("VBD", "AP"), require_copula),
    ("VP", ("VBZ", "NP"), require_copula),
    ("VP", ("VBD", "NP"), require_copula),

    ("VP", ("VP", "PP"), pass_head_features),
    ("VP", ("VP", "RB"), pass_head_features),
    ("VP", ("VP", "NP"), check_time_adjunct),

    # --- 4. PREPOSITIONS ---
    ("PP", ("IN", "NP"), lambda rhs: {'type': rhs[0]['feats'].get('type')}),
    ("PP", ("TO", "NP"), lambda rhs: {'type': 'dir'}),
]

## Functions to Convert CFG to CNF

In [ ]:
def convert_case1(grammar):
    """
    Handle rules that mix terminals and non-terminals (e.g., A -> B 'c').
    Replace terminals with dummy non-terminals.
    PRESERVES VALIDATORS.
    """
    new_grammar = Grammar()
    term_to_nonterm = {}

    for lhs, rules in grammar.rules_by_lhs.items():
        for rule in rules:
            rhs = rule['rhs']

            # 1. If rule is simple lexical (A -> 'w'), keep it exactly as is
            # We check if RHS is length 1 and if that symbol is considered terminal
            if len(rhs) == 1 and grammar.is_terminal(rhs[0]):
                new_grammar.add_rule(lhs, rhs,
                                     validator=rule['validator'],
                                     features=rule['features'])
                continue

            # 2. Check for mixed Terminals in Binary/Ternary rules
            new_rhs = []
            for symbol in rhs:
                if grammar.is_terminal(symbol):
                    # Create dummy non-terminal for the literal string
                    if symbol not in term_to_nonterm:
                        dummy = f"X_{symbol.upper()}"
                        term_to_nonterm[symbol] = dummy

                        # Add the dummy rule (X_THE -> 'the')
                        # No special features needed for the dummy wrapper itself
                        new_grammar.add_rule(dummy, (symbol,))

                    new_rhs.append(term_to_nonterm[symbol])
                else:
                    new_rhs.append(symbol)

            # Add the modified rule, preserving the original validator
            new_grammar.add_rule(lhs, tuple(new_rhs),
                                 validator=rule['validator'],
                                 features=rule['features'])

    return new_grammar

In [ ]:
def convert_case3(grammar):
    """
    Handle Long RHS (A -> B C D).
    Convert to binary: A -> X1 D, X1 -> B C.

    WARNING: This destroys 'validator' logic if the validator expects 3+ children.
    We preserve the validator ONLY if the rule is already binary/unary.
    """
    new_grammar = Grammar()

    def binarize(lhs, rhs, original_validator, original_features):
        """
        Recursive helper.
        """
        if len(rhs) <= 2:
            # Base case: Rule is short enough. Keep validator/features.
            return [{
                'lhs': lhs,
                'rhs': rhs,
                'validator': original_validator,
                'features': original_features
            }]

        # Recursive Step
        last_element = rhs[-1]
        remaining = rhs[:-1]

        # Create Dummy Symbol for the prefix
        new_symbol = f"X_{'_'.join(remaining)}"

        # 1. Create the Top Rule (LHS -> New_Symbol Last_Element)
        # We cannot safely apply the original validator here because it might expect 3 items.
        # We pass None.
        current_rule = {
            'lhs': lhs,
            'rhs': (new_symbol, last_element),
            'validator': None, # LOST VALIDATOR
            'features': original_features # Keep features on the head if applicable
        }

        # 2. Recurse on the remaining part
        # The children are intermediate nodes, they usually don't need the top-level features.
        child_rules = binarize(new_symbol, remaining, None, None)

        return [current_rule] + child_rules

    for lhs, rules in grammar.rules_by_lhs.items():
        for rule in rules:
            rhs = rule['rhs']

            # Generate binary versions
            generated_rules = binarize(lhs, rhs, rule['validator'], rule['features'])

            for r in generated_rules:
                new_grammar.add_rule(r['lhs'], r['rhs'],
                                     validator=r['validator'],
                                     features=r['features'])

    return new_grammar

In [ ]:
def convert_to_cnf(grammar):
    """
    Pipeline for CKY compatibility.
    Skipped Case 2 (Unit Rules) because the new CKY parser handles them natively.
    """
    # 1. Handle Mixed Terminals
    g1 = convert_case1(grammar)

    # 2. Binarize (Necessary for CKY logic)
    g3 = convert_case3(g1)

    return g3

## CKY Parser

In [ ]:
import collections

def cky_parse(words, grammar, debug=True):
    n = len(words)
    table = [[[] for _ in range(n + 1)] for _ in range(n + 1)]
    backpointers = collections.defaultdict(list)

    # --- 1. FILL DIAGONALS ---
    for j in range(1, n + 1):
        word = words[j - 1]
        lexical_rules = grammar.get_rules_for_rhs((word,))

        if debug and not lexical_rules:
            print(f"[DEBUG] No rule found for terminal '{word}'")

        for rule in lexical_rules:
            node = {
                'symbol': rule['lhs'],
                'feats': rule['features'] if rule['features'] else {},
                'start': j-1, 'end': j, 'is_terminal': True, 'word': word
            }
            table[j - 1][j].append(node)
            backpointers[(j - 1, j, rule['lhs'])].append((-1, word, None))
            if debug: print(f"[DEBUG] Added Terminal: {rule['lhs']} -> '{word}' {node['feats']}")

    # --- 2. MAIN LOOP ---
    for length in range(1, n + 1):
        for i in range(n - length + 1):
            j = i + length

            if length > 1:
                for k in range(i + 1, j):
                    left_cell = table[i][k]
                    right_cell = table[k][j]

                    for left_node in left_cell:
                        for right_node in right_cell:
                            # Look for Binary Rules
                            potential_rules = grammar.get_rules_for_rhs((left_node['symbol'], right_node['symbol']))

                            for rule in potential_rules:
                                validator = rule['validator']
                                new_features = None

                                if validator:
                                    new_features = validator([left_node, right_node])
                                    if debug and new_features is None:
                                        print(f"[DEBUG] Validation Failed for {rule['lhs']} -> {left_node['symbol']} {right_node['symbol']}")
                                        print(f"   Left Feats: {left_node['feats']}")
                                        print(f"   Right Feats: {right_node['feats']}")
                                else:
                                    new_features = {}

                                if new_features is not None:
                                    lhs = rule['lhs']
                                    # Duplicate Check
                                    if not any(n['symbol'] == lhs and n['feats'] == new_features for n in table[i][j]):
                                        new_node = {'symbol': lhs, 'feats': new_features, 'start': i, 'end': j, 'is_terminal': False}
                                        table[i][j].append(new_node)
                                        backpointers[(i, j, lhs)].append((k, left_node, right_node))
                                        if debug: print(f"[DEBUG] Created Node: {lhs} from {left_node['symbol']} + {right_node['symbol']} {new_features}")

            # --- 3. UNARY PASS ---
            added_new = True
            while added_new:
                added_new = False
                current_nodes = list(table[i][j])

                for child_node in current_nodes:
                    potential_rules = grammar.get_rules_for_rhs((child_node['symbol'],))

                    for rule in potential_rules:
                        validator = rule['validator']
                        new_features = validator([child_node]) if validator else child_node['feats'].copy()

                        if new_features is not None:
                            lhs = rule['lhs']
                            if not any(n['symbol'] == lhs and n['feats'] == new_features for n in table[i][j]):
                                new_node = {'symbol': lhs, 'feats': new_features, 'start': i, 'end': j, 'is_terminal': False}
                                table[i][j].append(new_node)
                                backpointers[(i, j, lhs)].append((-2, child_node, None))
                                added_new = True
                                if debug: print(f"[DEBUG] Created Unary: {lhs} -> {child_node['symbol']} {new_features}")

    # --- 4. TREE RECONSTRUCTION ---
    valid_starts = [node for node in table[0][n] if node['symbol'] == 'S']
    if valid_starts:
        return get_parse_tree(valid_starts[0], backpointers)
    else:
        if debug: print("[DEBUG] No 'S' node found at root.")
        return None

In [ ]:
def get_parse_tree(current_node, backpointers):
    """
    Recursive function to build a tuple-tree from nodes.
    Format: ('S', ('NP', 'I'), ('VP', 'bought...'))
    """

    # 1. Base Case: Terminal
    if current_node.get('is_terminal'):
        return (current_node['symbol'], current_node['word'])

    i, j = current_node['start'], current_node['end']
    symbol = current_node['symbol']

    # Get options from backpointers
    options = backpointers.get((i, j, symbol), [])

    if not options:
        return (symbol, "ERROR")

    # Pick the first valid derivation (Greedy)
    # Note: In a robust system, you might check which option matches current_node['feats']
    # For now, we take the first split.
    split_k, left_child, right_child = options[0]

    # 2. Case: Unary Rule (marked as -2)
    if split_k == -2:
        child_tree = get_parse_tree(left_child, backpointers)
        return (symbol, child_tree)

    # 3. Case: Binary Rule
    left_tree = get_parse_tree(left_child, backpointers)
    right_tree = get_parse_tree(right_child, backpointers)

    return (symbol, left_tree, right_tree)

## Back to CFG

In [ ]:
def unbinarize(tree):
    if isinstance(tree, tuple) and len(tree) == 2 and isinstance(tree[1], str):
        return tree
    label = tree[0]
    children = tree[1:]
    new_children = []
    for child in children:
        fixed_child = unbinarize(child)
        if fixed_child[0].startswith("X_"):
            new_children.extend(fixed_child[1:])
        else:
            new_children.append(fixed_child)
    return (label, *new_children)

## Test

In [ ]:
import sys

In [ ]:
import sys

def run_tests_and_save():
    """
    Runs the parser on:
    1. Project Valid Sentences (Must Pass)
    2. Project Invalid Sentences (Must Fail)
    3. New Robustness Sentences (Must Pass/Fail based on logic)

    Exports results to 'parser_results.txt'.
    """
    # 1. Initialize Pipeline
    grammar = Grammar()
    analyzer = MorphAnalyzer()

    # Load Lexicon & Rules
    analyzer.feed_to_grammar(grammar)
    grammar.fill_rules(SYNTACTIC_RULES)

    # Convert to CNF (using the new "safe" converter)
    cnf_grammar = convert_to_cnf(grammar)

    # --- TEST DATASETS ---

    # Set A: Required Valid Sentences (from Project Description)
    valid_sentences = [
        ["I", "bought", "a", "present", "for", "my", "friend", "yesterday"],
        ["I", "enjoy", "historical", "novels"],
        ["I", "helped", "my", "mother", "with", "dinner", "yesterday"],
        ["epics", "tell", "about", "our", "national", "culture", "and", "history"],
        ["watermelon", "is", "the", "most", "beautiful", "fruit", "of", "summer"],
        ["we", "watched", "the", "moonlight", "under", "this", "tree", "every", "night"],
        ["the", "school", "was", "quite", "far", "from", "our", "village"],
        ["will", "you", "attend", "the", "meeting", "tonight"],
        ["when", "did", "you", "come", "here", "lastly"],
        ["do", "not", "listen", "to", "loud", "music"]
    ]

    # Set B: Required Invalid Sentences (Agreement/Subcat Errors)
    invalid_sentences = [
        ["I", "buys", "a", "gift", "for", "my", "friend"],       # Agreement: 1SG vs 3SG
        ["I", "read", "a", "historical", "novels"],             # Agreement: Det(a) vs Noun(novels)
        ["I", "will", "help", "my", "father", "yesterday"],     # Tense Mismatch (Future + Past Adv) - *Advanced check
        ["I", "went", "the", "school"],                         # Subcat: Intransitive 'went' + NP
        ["I", "went", "in", "the", "school"],                   # Subcat: 'went' needs Directional (to/from), not Locative (in)
        ["I", "was", "read", "the", "book"]                     # Syntax: Aux 'was' + Base/Past Participle error
    ]

    # Set C: Robustness Tests (New Combinations of Existing Vocab)
    # These prove to the professor that your rules are generic (Feature-Based), not hardcoded.
    robustness_valid = [
        # Subject-Verb Agreement (3rd Person Singular)
        ["she", "buys", "a", "present"],
        # Subject-Verb Agreement (Plural)
        ["friends", "buy", "historical", "novels"],
        # Complex NP + Intransitive + PP
        ["my", "father", "went", "to", "the", "school"],
        # Modal Question with Possessive
        ["can", "my", "friend", "read", "the", "book"]
    ]

    robustness_invalid = [
        # Mismatch: 3SG Subject vs Plural Verb
        ["she", "buy", "a", "present"],
        # Mismatch: Plural Subject vs 3SG Verb
        ["friends", "buys", "historical", "novels"],
        # Missing Preposition for Intransitive
        ["she", "went", "dinner"]
    ]

    output_filename = "parser_results.txt"

    # --- EXECUTION ---
    with open(output_filename, "w", encoding="utf-8") as f:
        # Redirect stdout to file
        original_stdout = sys.stdout
        sys.stdout = f

        print("=== PARSER EVALUATION REPORT ===\n")

        # Helper to run a batch
        def run_batch(name, sentences, should_pass):
            print(f"--- {name} ---")
            count = 0
            for sent in sentences:
                sent_str = " ".join(sent)
                sent_lower = [w.lower() for w in sent]

                # Run CKY
                tree = cky_parse(sent_lower, cnf_grammar)

                # Check Result
                success = (tree is not None)

                # Validation Logic
                if should_pass:
                    if success:
                        count += 1
                        clean_tree = unbinarize(tree)
                        print(f"Sentence: {sent_str}")
                        print(f"Result:   [PASS] (Tree generated)")
                        print(f"Tree:     {clean_tree}\n")
                    else:
                        print(f"Sentence: {sent_str}")
                        print(f"Result:   [FAIL] (No Parse Found)\n")
                else:
                    if not success:
                        count += 1
                        print(f"Sentence: {sent_str}")
                        print(f"Result:   [PASS] (Correctly Rejected)\n")
                    else:
                        print(f"Sentence: {sent_str}")
                        print(f"Result:   [FAIL] (Incorrectly Accepted!)\n")
            return count

        # Run Batches
        s1 = run_batch("Part 1: Required Valid Sentences", valid_sentences, True)
        s2 = run_batch("Part 2: Required Invalid Sentences", invalid_sentences, False)
        s3 = run_batch("Part 3: Robustness (Generalization Check - Valid)", robustness_valid, True)
        s4 = run_batch("Part 4: Robustness (Generalization Check - Invalid)", robustness_invalid, False)

        # Final Scores
        total_valid = len(valid_sentences) + len(robustness_valid)
        total_invalid = len(invalid_sentences) + len(robustness_invalid)
        pass_valid = s1 + s3
        pass_invalid = s2 + s4

        print("====================================")
        print("FINAL STATISTICS")
        print("====================================")
        print(f"Valid Sentences Accepted:   {pass_valid}/{total_valid} ({(pass_valid/total_valid)*100:.1f}%)")
        print(f"Invalid Sentences Rejected: {pass_invalid}/{total_invalid} ({(pass_invalid/total_invalid)*100:.1f}%)")
        print("====================================")

        # Restore stdout
        sys.stdout = original_stdout

    print(f"Test complete. Results written to {output_filename}")

In [ ]:
run_tests_and_save()

Test complete. Results written to parser_results.txt


In [ ]:
# Setup
grammar = Grammar()
analyzer = MorphAnalyzer()
analyzer.feed_to_grammar(grammar)
grammar.fill_rules(SYNTACTIC_RULES)
cnf_grammar = convert_to_cnf(grammar)

# Debug Test
test_sent = ["i", "bought", "a", "present"] # Lowercase explicit
print(f"Testing: {test_sent}")
tree = cky_parse(test_sent, cnf_grammar, debug=True)

if tree:
    print("\nSUCCESS:")
    print(unbinarize(tree))
else:
    print("\nFAILURE")

Testing: ['i', 'bought', 'a', 'present']
[DEBUG] Added Terminal: PRP -> 'i' {'num': 'sg', 'per': 1, 'case': 'nom'}
[DEBUG] Added Terminal: VBD -> 'bought' {'tense': 'past', 'subcat': 'tr'}
[DEBUG] Added Terminal: DT -> 'a' {'num': 'sg'}
[DEBUG] Added Terminal: NN -> 'present' {'num': 'sg', 'per': 3}
[DEBUG] Created Unary: NP -> PRP {'num': 'sg', 'per': 1, 'case': 'nom'}
[DEBUG] Created Unary: NP -> NN {'num': 'sg', 'per': 3}
[DEBUG] Created Node: NP from DT + NN {'num': 'sg', 'per': 3, 'sem': None}
[DEBUG] Created Node: VP from VBD + NP {'tense': 'past', 'num': None, 'per': None}
[DEBUG] Validation Failed for VP -> VBD NP
   Left Feats: {'tense': 'past', 'subcat': 'tr'}
   Right Feats: {'num': 'sg', 'per': 3, 'sem': None}
[DEBUG] Created Node: S from NP + VP {'type': 'decl', 'tense': 'past'}
[DEBUG] Created Node: S_SUBJ_VP from NP + VP {'type': 'decl', 'tense': 'past'}

SUCCESS:
('S', ('NP', ('PRP', 'i')), ('VP', ('VBD', 'bought'), ('NP', ('DT', 'a'), ('NN', 'present'))))
